# 03 — Bar Backtest & Analytics

Picks up where notebook 02 left off: runs `BarBacktest` on the MA-200 signal columns,
then measures performance with `SignalAnalytics`.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalAnalytics
      (bars)            (signal cols)   (return cols)   (stats + charts)
```

In [1]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalAnalytics
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Data → Signal → Backtest

In [2]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

# Fetch warmup bars before start so MA-200 is valid from day one
fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)

df = bt_result.data
print(f"{df.shape[0]:,} bars  |  {df.index.get_level_values('symbol').nunique()} symbols")

2026-04-27 01:17:17.515 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=de1ec4900549


2,193 bars  |  3 symbols


In [3]:
bt_result.data.columns

Index(['open', 'high', 'low', 'close', 'volume', 'ma', 'signal_close',
       'signal_open', 'trade_direction', 'turnover', 'enter', 'exit', 'cycle',
       'signal_age', 'return_mark_to_close', 'return_conservative',
       'return_net', 'position_start', 'position_end',
       'strategy_equity_mark_to_close', 'strategy_equity_conservative',
       'strategy_equity_net', 'trade_cycle_id'],
      dtype='str')

In [4]:
sym =  "BTC-USD"
bt_result.data.query("symbol == @sym").unstack('symbol').to_clipboard()

## 2. Execution Columns

`BarBacktest` appends these columns to the signal DataFrame:

| Column | Description |
|---|---|
| `return_mark_to_close` | Fill-aware daily return — best-case timing (open on `init`, open on `exit`) |
| `return_conservative` | Worst-case timing (high on `init`, low on `exit`) |
| `return_net` | MTC minus `cost_bps/10 000` on entry and exit bars |
| `position_start` | 1 when capital is deployed at the bar open |
| `position_end` | 1 when capital is deployed at the bar close |
| `strategy_equity_mark_to_close` | Cumulative product of MTC returns (starts at 1.0) |
| `strategy_equity_conservative` | Cumulative product of conservative returns |
| `strategy_equity_net` | Cumulative product of net returns |
| `trade_cycle_id` | Integer ID incrementing on each new trade or flat period |

In [5]:
exec_cols = [
    "cycle", "return_mark_to_close", "return_conservative", "return_net",
    "position_start", "position_end",
    "strategy_equity_mark_to_close", "strategy_equity_conservative", "strategy_equity_net",
    "trade_cycle_id",
]
df[exec_cols].head(12)

cycle  return_mark_to_close  return_conservative  \
symbol  timestamp                                                     
BTC-USD 2022-01-01  None                   0.0                  0.0   
        2022-01-02  None                   0.0                  0.0   
        2022-01-03  None                   0.0                  0.0   
        2022-01-04  None                   0.0                  0.0   
        2022-01-05  None                   0.0                  0.0   
        2022-01-06  None                   0.0                  0.0   
        2022-01-07  None                   0.0                  0.0   
        2022-01-08  None                   0.0                  0.0   
        2022-01-09  None                   0.0                  0.0   
        2022-01-10  None                   0.0                  0.0   
        2022-01-11  None                   0.0                  0.0   
        2022-01-12  None                   0.0                  0.0   

                    return_net  position_start  position_end  \
symbol  timestamp                                              
BTC-USD 2022-01-01         0.0               0             0   
        2022-01-02         0.0               0             0   
        2022-01-03         0.0               0             0   
        2022-01-04         0.0               0             0   
        2022-01-05         0.0               0             0   
        2022-01-06         0.0               0             0   
        2022-01-07         0.0               0             0   
        2022-01-08         0.0               0             0   
        2022-01-09         0.0               0             0   
        2022-01-10         0.0               0             0   
        2022-01-11         0.0               0             0   
        2022-01-12         0.0               0             0   

                    strategy_equity_mark_to_close  \
symbol  timestamp                                   
BTC-USD 2022-01-01                            1.0   
        2022-01-02                            1.0   
        2022-01-03                            1.0   
        2022-01-04                            1.0   
        2022-01-05                            1.0   
        2022-01-06                            1.0   
        2022-01-07                            1.0   
        2022-01-08                            1.0   
        2022-01-09                            1.0   
        2022-01-10                            1.0   
        2022-01-11                            1.0   
        2022-01-12                            1.0   

                    strategy_equity_conservative  strategy_equity_net  \
symbol  timestamp                                                       
BTC-USD 2022-01-01                           1.0                  1.0   
        2022-01-02                           1.0                  1.0   
        2022-01-03                           1.0                  1.0   
        2022-01-04                           1.0                  1.0   
        2022-01-05                           1.0                  1.0   
        2022-01-06                           1.0                  1.0   
        2022-01-07                           1.0                  1.0   
        2022-01-08                           1.0                  1.0   
        2022-01-09                           1.0                  1.0   
        2022-01-10                           1.0                  1.0   
        2022-01-11                           1.0                  1.0   
        2022-01-12                           1.0                  1.0   

                    trade_cycle_id  
symbol  timestamp                   
BTC-USD 2022-01-01               1  
        2022-01-02               1  
        2022-01-03               1  
        2022-01-04               1  
        2022-01-05               1  
        2022-01-06               1  
        2022-01-07               1  
        2022-01-08          

## 3. Summary Statistics

In [6]:
analytics = SignalAnalytics(bt_result)

summary = analytics.summary()
summary.style \
    .format({
        "return_mtc":          "{:+.1%}",
        "return_conservative":  "{:+.1%}",
        "pct_invested":         "{:.1f}%",
    }) \
    .background_gradient(subset=["return_mtc", "return_conservative"], cmap="RdYlGn") \
    .set_caption("MA-200 Trend Signal — Period Summary")

,return_mtc,return_conservative,return_net,entries,exits,invested_days,pct_invested
symbol,,,,,,,
BTC-USD,+104.0%,+74.4%,1.039914,3,2,296,40.5%
ETH-USD,+30.0%,-1.1%,0.299747,5,4,296,40.5%
SOL-USD,+180.3%,+42.9%,1.803328,9,8,223,30.5%


## 4. Equity Curves

In [7]:
eq_mtc = analytics.equity(method="mtc")
eq_con = analytics.equity(method="conservative")

colours = [PALETTE["accent_blue"], PALETTE["accent_green"], PALETTE["accent_purple"]]

fig = go.Figure()
for i, sym in enumerate(eq_mtc.columns):
    c = colours[i % len(colours)]
    fig.add_trace(go.Scatter(
        x=eq_mtc.index, y=eq_mtc[sym],
        name=f"{sym} MTC",
        line=dict(color=c, width=1.8),
    ))
    fig.add_trace(go.Scatter(
        x=eq_con.index, y=eq_con[sym],
        name=f"{sym} Conservative",
        line=dict(color=c, width=1, dash="dot"),
        showlegend=True,
    ))

apply_theme(fig, title="Equity Curves — MA-200 Trend Signal (MTC vs Conservative)", height=460)
fig.update_layout(yaxis_title="Equity (normalised to 1.0)")
fig.show()

## 5. Drawdown

In [8]:
def drawdown(equity: pd.DataFrame) -> pd.DataFrame:
    running_max = equity.cummax()
    return equity / running_max - 1

dd = drawdown(eq_mtc)

fill_colours = ["rgba(88,166,255,0.12)", "rgba(63,185,80,0.12)", "rgba(163,113,247,0.12)"]

fig = go.Figure()
for i, sym in enumerate(dd.columns):
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd[sym],
        name=sym,
        fill="tozeroy",
        line=dict(color=colours[i % len(colours)], width=1),
        fillcolor=fill_colours[i % len(fill_colours)],
    ))

apply_theme(fig, title="Drawdown — MA-200 Trend Signal (MTC)", height=380)
fig.update_layout(yaxis_title="Drawdown", yaxis_tickformat=".0%")
fig.show()

## 6. Trade Duration Distribution

In [9]:
# Bars per trade (each trade_cycle_id where position was held)
trade_durations = (
    df[df["position_end"] == 1]
    .reset_index(level="symbol")
    .groupby(["symbol", "trade_cycle_id"])
    .size()
    .reset_index(name="duration_bars")
)

fig = go.Figure()
for i, sym in enumerate(trade_durations["symbol"].unique()):
    durations = trade_durations.loc[trade_durations["symbol"] == sym, "duration_bars"]
    fig.add_trace(go.Histogram(
        x=durations,
        name=sym,
        marker_color=colours[i % len(colours)],
        opacity=0.75,
        nbinsx=20,
    ))

apply_theme(fig, title="Trade Duration Distribution (bars held)", height=380)
fig.update_layout(
    xaxis_title="Duration (bars)",
    yaxis_title="Count",
    barmode="overlay",
)
fig.show()

## 7. MTC vs Conservative — Final Return Comparison

In [10]:
syms = summary.index.tolist()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=syms,
    y=summary["return_mtc"] * 100,
    name="Mark-to-Close",
    marker_color=PALETTE["accent_blue"],
))
fig.add_trace(go.Bar(
    x=syms,
    y=summary["return_conservative"] * 100,
    name="Conservative",
    marker_color=PALETTE["accent_orange"],
))

apply_theme(fig, title="Total Return — MTC vs Conservative", height=380)
fig.update_layout(
    barmode="group",
    yaxis_title="Total Return (%)",
    yaxis_tickformat=".0f",
)
fig.show()

## 8. Combined Portfolio Analysis

`portfolio_equity()` weights each in-signal symbol at `1 / N_universe` at each bar —
the same fixed fraction used by the engine path, so uninvested capital sits as cash
when fewer symbols are in signal.  `portfolio_metrics()` returns a full
`PerformanceMetrics` object (Sharpe, Calmar, VaR, …).

In [11]:
port_eq = analytics.portfolio_equity(method="conservative")

fig = go.Figure()

# Individual symbols (muted, dotted)
for i, sym in enumerate(eq_con.columns):
    fig.add_trace(go.Scatter(
        x=eq_con.index, y=eq_con[sym],
        name=sym,
        line=dict(color=colours[i % len(colours)], width=1, dash="dot"),
        opacity=0.5,
    ))

# Equal-weight portfolio (bold)
fig.add_trace(go.Scatter(
    x=port_eq.index, y=port_eq,
    name="Equal-weight Portfolio",
    line=dict(color=PALETTE["accent_yellow"], width=2.5),
))

apply_theme(fig, title="Equal-Weight Portfolio vs Individual Symbols (conservative)", height=460)
fig.update_layout(yaxis_title="Equity (normalised to 1.0)")
fig.show()

In [12]:
analytics.portfolio_metrics().summary()

Total Return       106.52%
CAGR                28.40%
Ann. Volatility     25.25%
Sharpe Ratio          1.12
Sortino Ratio         1.22
Calmar Ratio          1.25
Max Drawdown       -22.65%
Avg Drawdown       -12.70%
Skewness              0.83
Kurtosis              7.99
VaR 95%             -2.22%
CVaR 95%            -3.64%
Omega Ratio           1.34
Tail Ratio            1.30
dtype: str

## 9. Portfolio Capital Models — Side-by-Side

Three ways to aggregate individual symbol returns into a portfolio NAV:

| Model | Within a trade | Between trades | Use when |
|---|---|---|---|
| **Rebalanced** | Daily reset to `1/N_universe` weight | Geometric (through rebalancing) | Comparing signal quality independent of capital size |
| **Reinvested** | Compounds from `1/N_universe` entry | Geometric (winners grow, cash stays flat) | Long-only momentum with no rebalancing |
| **Fixed $1k/entry** | Compounds from `$amount_per_entry` | Arithmetic (P&L to cash, fresh stake each entry) | Fixed-size position sizing, take-profit on exit |

`amount_per_entry` is normalised: NAV is divided by `amount × N_universe` so all three curves start at 1.0 and are directly comparable.

In [ ]:
port_rebalanced = analytics.portfolio_equity(method="conservative", reinvest=False)
port_reinvested  = analytics.portfolio_equity(method="conservative", reinvest=True)
port_fixed       = analytics.portfolio_equity_fixed_amount(method="conservative", amount_per_entry=1_000)

fig = go.Figure()

# Individual symbols (muted background)
for i, sym in enumerate(eq_con.columns):
    fig.add_trace(go.Scatter(
        x=eq_con.index, y=eq_con[sym],
        name=sym, showlegend=True,
        line=dict(color=colours[i % len(colours)], width=1, dash="dot"),
        opacity=0.35,
    ))

fig.add_trace(go.Scatter(
    x=port_rebalanced.index, y=port_rebalanced,
    name="Rebalanced (1/N daily)",
    line=dict(color=PALETTE["accent_yellow"], width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=port_reinvested.index, y=port_reinvested,
    name="Reinvested (let it run)",
    line=dict(color=PALETTE["accent_green"], width=2),
))
fig.add_trace(go.Scatter(
    x=port_fixed.index, y=port_fixed,
    name="Fixed $1k/entry",
    line=dict(color=PALETTE["accent_blue"], width=2.5),
))

apply_theme(fig, title="Portfolio Capital Models — Conservative Fill", height=500)
fig.update_layout(yaxis_title="NAV (normalised to 1.0)")
fig.show()

In [15]:
def _metrics_row(pm) -> dict:
    return {
        "Total Return": f"{pm.total_return:+.1%}",
        "CAGR":         f"{pm.annualised_return:+.1%}",
        "Ann. Vol":     f"{pm.annualised_vol:.1%}",
        "Sharpe":       f"{pm.sharpe:.2f}",
        "Max Drawdown": f"{pm.max_drawdown:.1%}",
        "Calmar":       f"{pm.calmar:.2f}",
    }

rows = {
    "Rebalanced":      _metrics_row(analytics.portfolio_metrics(method="conservative", reinvest=False)),
    "Reinvested":      _metrics_row(analytics.portfolio_metrics(method="conservative", reinvest=True)),
    "Fixed $1k/entry": _metrics_row(analytics.portfolio_metrics_fixed_amount(method="conservative", amount_per_entry=1_000)),
}

pd.DataFrame(rows).T

,Total Return,CAGR,Ann. Vol,Sharpe,Max Drawdown,Calmar
Rebalanced,+43.3%,+13.2%,25.4%,0.61,-35.5%,0.37
Reinvested,+38.9%,+12.0%,23.9%,0.59,-29.3%,0.41
Fixed $1k/entry,+106.9%,+28.5%,35.8%,0.88,-44.0%,0.65
